# peer_hooks

> The hooks that make a Claude Code or Codex session a peer: they register it, give it its messages, and guard other sessions' folders.

In [ ]:
#| default_exp peer_hooks

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import os, tempfile
from unittest.mock import patch
from fastcore.test import test_eq

Claude Code and Codex have no control socket, as Ramabana has. They run hooks: commands that the agent starts at fixed events. Every peer hook is one command, `ramabana peer-hook KIND EVENT`. It reads the hook input as JSON on stdin and writes what the agent expects.

- `session-start` registers the session and tells the agent its peer name, the live peers and how to send.
- `prompt` gives the agent its unread messages when the user types. It also ends work together mode, because the user is back, and shows the report of the mode.
- `stop` keeps the agent going while messages are unread.
- `wait` (Claude Code only) waits in the background and wakes an idle session when a message comes.
- `end` removes the registration.
- `pre-tool` refuses a write into a folder that another live session holds.

Hooks must be fast and must not break the agent. They make no model call and use no network. An error goes to `hooks.log` in the peers folder, and the hook then does nothing. The module does not load the agent code.

In [ ]:
#| export
import fcntl, json, os, re, subprocess, sys, time, traceback
from contextlib import suppress
from dataclasses import dataclass
from pathlib import Path
from ramabana.peers import (agent_pid, all_peers, folder_denial, git_root, is_partner, log, peer_lines, peer_text, peers_dir, pid_alive,
                            read_log, register, take, together_off, together_state, unregister, wait_take)
from ramabana.peer_report import report_path

## Adapters

An `Adapter` holds what is different for each agent kind: its config file, the file that gets the `peer-work` skill, the variable that holds its session id, and whether a hook can wake it. `ramabana setup peers` uses them to install the hooks.

In [ ]:
#| export
@dataclass
class Adapter:
    "What the peer hooks need to know about one agent kind."
    kind: str
    config: str
    skill_path: str
    session_env: str|None
    wakes: bool

ADAPTERS = {'claude': Adapter('claude', '~/.claude/settings.json', '~/.claude/skills/peer-work/SKILL.md', 'CLAUDE_CODE_SESSION_ID', True),
            'codex': Adapter('codex', '~/.codex/hooks.json', '~/.codex/AGENTS.md', None, False)}

`hook_block` gives the hook entries for one agent's config, by event. Claude Code gets a second `Stop` hook, the waiter: `async` runs it in the background, and `asyncRewake` lets its exit code 2 wake an idle session. Claude Code stops an `asyncRewake` hook at its `timeout`, so the waiter gets a long one. Codex has no way to wake an idle session, and it names its file edit tool `apply_patch`.

In [ ]:
#| export
WAIT_TIMEOUT = 7*24*3600   #: seconds before Claude Code stops the waiter; the next `Stop` starts a new one
EDIT_TOOLS = {'claude': 'Edit|Write|NotebookEdit', 'codex': 'apply_patch'}

def _group(kind, event, matcher='', **kw):
    "One hook group that runs `ramabana peer-hook KIND EVENT`."
    return ({'matcher': matcher} if matcher else {}) | {'hooks': [{'type': 'command', 'command': f'ramabana peer-hook {kind} {event}', **kw}]}

def hook_block(kind):
    "The hook groups, by event, that agent `kind` needs in its config."
    wake = [_group(kind, 'wait', **{'async': True, 'asyncRewake': True, 'timeout': WAIT_TIMEOUT})] if ADAPTERS[kind].wakes else []
    return {'SessionStart': [_group(kind, 'session-start')], 'UserPromptSubmit': [_group(kind, 'prompt')], 'Stop': [_group(kind, 'stop'), *wake],
            'SessionEnd': [_group(kind, 'end')], 'PreToolUse': [_group(kind, 'pre-tool', EDIT_TOOLS[kind])]}

Each event has one group. Claude Code has two `Stop` groups; Codex has one.

In [ ]:
b = hook_block('claude')
test_eq(b['PreToolUse'], [{'matcher': 'Edit|Write|NotebookEdit', 'hooks': [{'type': 'command', 'command': 'ramabana peer-hook claude pre-tool'}]}])
test_eq([g['hooks'][0].get('asyncRewake') for g in b['Stop']], [None, True])
test_eq(len(hook_block('codex')['Stop']), 1)

## What the agent reads

`_messages` writes unread messages for the agent with `peer_text`. An ask also says who decides. Outside work together mode, the agent must ask the user first. In work together mode with that sender, it can act without the user. The ask also shows the command that answers it. Codex does not give its session id to commands, so its commands carry `--session`.

In [ ]:
#| export
ASK_USER = 'Ask the user before you act on this ask.'
ASK_FREE = 'Work together mode is on with this sender. You can act on this ask without the user.'

def _key(ad, data): return f"{ad.kind}-{data['session_id']}"

def _flag(ad, data):
    "The `--session` option for an agent that does not give its session id to commands, else ''."
    return '' if ad.session_env else f" --session {data['session_id']}"

def _messages(ad, d, data, got):
    "The messages `got` as text for the agent; an ask says who decides and how to answer."
    st = together_state(d, _key(ad, data))
    def one(e):
        if e.kind != 'ask': return peer_text(e)
        who = ASK_FREE if st and is_partner(st, e.frm) else ASK_USER
        return f"{peer_text(e)}\n{who} Answer with `ramabana tell{_flag(ad, data)} {e.frm.get('key', '?')} \"...\" --reply-to {e.thread}`."
    return '\n\n'.join(map(one, got))

def _context(event, text):
    "The hook result that adds `text` to the agent's context."
    return 0, json.dumps({'hookSpecificOutput': {'hookEventName': event, 'additionalContext': text}}), ''

## Session start

`_start` registers the session with the git root of its folder and the agent's process id. Then it tells the agent who it is, which peers are live, and how to send.

In [ ]:
#| export
BRIEF = """You are a peer session. Claude Code, Codex and Ramabana sessions on this Mac can send you notes and asks, and you can send them yours.
Your peer name is {name}. Your key is {key}.
Live peers:
{peers}
Send a note: `ramabana tell{s} TARGET "text"`. Send an ask: add `--ask --repo R`. Answer an ask: add `--reply-to THREAD`. TARGET is a number from `ramabana peers{s}`, a session id or a folder name.
Read the `peer-work` skill before you work for a peer or ask a peer to work."""
NO_SESSION_ENV = 'Your agent does not give your session id to commands. Add `--session {session}` after the command name in each `ramabana peers`, `tell`, `inbox` and `together` command.'

def _start(ad, d, data, cfg):
    "Register the session and brief the agent."
    row = register(d, ad.kind, data['session_id'], [git_root(data.get('cwd') or '.')], agent_pid())
    text = BRIEF.format(name=row['name'], key=row['key'], peers=peer_lines(all_peers(cfg, exclude=row['key'], timeout=1)) or 'none', s=_flag(ad, data))
    return _context('SessionStart', text + ('' if ad.session_env else '\n' + NO_SESSION_ENV.format(session=data['session_id'])))

## The user is back

When the user types, `_ended` ends work together mode. It returns the start time of a mode that ended since the user last saw one, also a mode that ended while the user was away. A `report_shown` row in the log makes sure the user sees each end once. `_report_note` gives the saved report, or tells the agent to make it.

In [ ]:
#| export
def _ended(d, key):
    "End work together mode of `key`; the start time of a mode that ended and that the user has not seen, else None."
    if st := together_off(d, key): log(d, {'event': 'together_end', 'key': key, 'started': st['started'], 'at': time.time()})
    ends = [r for r in read_log(d) if r.get('key') == key and r.get('event') in ('together_end', 'report_shown')]
    if not ends or ends[-1]['event'] == 'report_shown': return None
    log(d, {'event': 'report_shown', 'key': key, 'started': ends[-1]['started'], 'at': time.time()})
    return ends[-1]['started']

def _report_note(since, cfg):
    "The saved report of the mode that started at `since`, or how to make it."
    p = report_path(since, cfg)
    if p.exists(): return f'Work together mode ended. Show the user this report, saved at {p}:\n\n{p.read_text()}'
    return f'Work together mode ended. Run `ramabana report --since {since}` and show the user the report.'

def _prompt(ad, d, data, cfg):
    "End work together mode, and give the agent its unread messages and the report of the mode that ended."
    since = _ended(d, _key(ad, data))
    got = take(d, _key(ad, data))
    parts = [*([_messages(ad, d, data, got)] if got else []), *([_report_note(since, cfg)] if since is not None else [])]
    return _context('UserPromptSubmit', '\n\n'.join(parts)) if parts else (0, '', '')

## Stop and wait

`_stop` blocks the stop while messages are unread, so the agent goes on and reads them. The reason is the messages. It blocks only for new messages, so it cannot loop.

`_wait` is the background waiter. Only one runs for each session: it holds a lock on the `.wait` file and writes its process id there. A second waiter exits at once. A lock dies with its process, so a crashed waiter does not block the next one. On messages, the waiter exits with code 2 and writes them to stderr, which wakes Claude. It exits with code 0 when the session ends. When work together mode expires while it waits, it saves the report first.

In [ ]:
#| export
def _stop(ad, d, data, cfg):
    "Block the stop while messages are unread."
    got = take(d, _key(ad, data))
    return (0, json.dumps({'decision': 'block', 'reason': _messages(ad, d, data, got)}), '') if got else (0, '', '')

def _live(d, key):
    "True while the registration of `key` exists and its agent process lives."
    try: return pid_alive(json.loads((d/f'{key}.json').read_text())['pid'])
    except (OSError, ValueError, KeyError): return False

def _save_report(since):
    "Save the report of the mode that started at `since` with `ramabana report --save`; a failure is logged."
    try: subprocess.run(['ramabana', 'report', '--save', '--since', str(since)], capture_output=True, timeout=600)
    except (OSError, subprocess.SubprocessError): _log_error('report')

def _watch(d, key):
    "The `alive` test of a waiter: true while the session lives; when work together mode expires, the report is saved first."
    seen = together_state(d, key)
    def alive():
        nonlocal seen
        st = together_state(d, key)
        if seen and st is None and time.time() >= seen['until']: _save_report(seen['started'])
        seen = st
        return _live(d, key)
    return alive

def _wait(ad, d, data, cfg):
    "Wait for a message; exit 2 with it on stderr, or 0 at once when another waiter runs."
    key = _key(ad, data)
    with open(d/f'{key}.wait', 'a+') as f:
        try: fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError: return 0, '', ''
        f.truncate(0); f.write(str(os.getpid())); f.flush()
        got = wait_take(d, key, None, alive=_watch(d, key))
    return (2, '', _messages(ad, d, data, got)) if got else (0, '', '')

## Session end and the folder guard

`_end` removes the registration and the `.wait` file, so the waiter stops. Claude Code gives `SessionEnd` only 1.5 s, so it does nothing more.

`_pre_tool` finds the paths a tool call writes. Claude Code's file tools give `file_path` or `notebook_path`. Codex's `apply_patch` gives the patch, which names each file. `folder_denial` checks each path. A refusal names the owner, and the agent can send it a message instead.

In [ ]:
#| export
PATCH_PATH = re.compile(r'^\*\*\* (?:(?:Add|Update|Delete) File|Move to): (.+?)\s*$', re.M)

def _end(ad, d, data, cfg):
    "Remove the registration and the waiter's lock file."
    unregister(d, ad.kind, data['session_id'])
    (d/f'{_key(ad, data)}.wait').unlink(missing_ok=True)
    return 0, '', ''

def _paths(data):
    "The paths that the tool call in `data` writes."
    ti = data.get('tool_input') or {}
    return [p for k in ('file_path', 'notebook_path') if (p := ti.get(k))] + (PATCH_PATH.findall(ti.get('command', '')) if data.get('tool_name') == 'apply_patch' else [])

def _pre_tool(ad, d, data, cfg):
    "Refuse a write into a folder that another live session holds."
    if not (paths := _paths(data)): return 0, '', ''
    rows, cwd = all_peers(cfg, timeout=1), Path(data.get('cwd') or '.')
    why = next(filter(None, (folder_denial(cwd/p, _key(ad, data), rows) for p in paths)), None)
    out = {'hookSpecificOutput': {'hookEventName': 'PreToolUse', 'permissionDecision': 'deny', 'permissionDecisionReason': why}}
    return (0, json.dumps(out), '') if why else (0, '', '')

A patch names files to add, change, delete and move to.

In [ ]:
test_eq(_paths({'tool_name': 'apply_patch', 'tool_input': {'command': '*** Begin Patch\n*** Update File: a.py\n*** Move to: b.py\n*** Add File: /r/c.py\n*** End Patch'}}), ['a.py', 'b.py', '/r/c.py'])
test_eq(_paths({'tool_name': 'Bash', 'tool_input': {'command': '*** Add File: x'}}), [])
test_eq(_paths({'tool_input': {'notebook_path': '/r/n.ipynb'}}), ['/r/n.ipynb'])

## The entry point

`peer_hook` runs one event and returns the exit code, stdout and stderr. An error is logged to `hooks.log` and gives exit code 0 with no output, so a broken hook never stops the agent. `main` is `ramabana peer-hook`; `python -m ramabana.peer_hooks` runs it too.

In [ ]:
#| export
EVENTS = {'session-start': _start, 'prompt': _prompt, 'stop': _stop, 'wait': _wait, 'end': _end, 'pre-tool': _pre_tool}

def _log_error(what, d=None):
    "Add the current traceback to `hooks.log`."
    with suppress(OSError), open(Path(d or peers_dir())/'hooks.log', 'a') as f: f.write(f'{time.ctime()} {what}\n{traceback.format_exc()}\n')

def peer_hook(kind, event, data, d=None, cfg=None):
    "Run hook `event` of agent `kind` on the hook input `data`: (exit code, stdout, stderr); an error is logged and gives (0, '', '')."
    try:
        d = Path(d) if d else peers_dir(cfg)
        return EVENTS[event](ADAPTERS[kind], d, data, cfg)
    except Exception:
        _log_error(f'{kind} {event}', d or peers_dir(cfg))
        return 0, '', ''

def main(argv):
    "Run `ramabana peer-hook KIND EVENT` on the hook input in stdin; the exit code."
    kind, event = [*argv, '', ''][:2]
    try: data = json.loads(sys.stdin.read() or '{}')
    except ValueError: data = {}
    code, out, err = peer_hook(kind, event, data)
    sys.stdout.write(out); sys.stderr.write(err)
    return code

In [ ]:
#| export
#| eval: false
if __name__ == '__main__': sys.exit(main(sys.argv[1:]))

A stop with nothing unread lets the agent stop. A note makes it block, once. A write into another session's folder is refused, and a write into the session's own folder is not. A broken input is logged.

In [ ]:
from ramabana.peers import Envelope, post, register
with tempfile.TemporaryDirectory() as t, patch.dict(os.environ, RAMABANA_PEERS=t):
    s1 = {'session_id': 's1', 'cwd': t}
    test_eq(peer_hook('claude', 'stop', s1, cfg=t), (0, '', ''))
    post(t, 'claude-s1', Envelope('note', 'v2 is out', {'key': 'ramabana-r1', 'name': 'leela'}))
    assert 'v2 is out' in json.loads(peer_hook('claude', 'stop', s1, cfg=t)[1])['reason']
    test_eq(peer_hook('claude', 'stop', s1, cfg=t), (0, '', ''))
    register(t, 'codex', 'c9', ['/r/abhyasa'], os.getpid())
    peer_hook('claude', 'session-start', s1, cfg=t)
    deny = json.loads(peer_hook('claude', 'pre-tool', s1 | {'tool_name': 'Edit', 'tool_input': {'file_path': '/r/abhyasa/a.py'}}, cfg=t)[1])
    test_eq(deny['hookSpecificOutput']['permissionDecision'], 'deny')
    test_eq(peer_hook('claude', 'pre-tool', s1 | {'tool_name': 'Edit', 'tool_input': {'file_path': f'{t}/a.py'}}, cfg=t), (0, '', ''))
    test_eq(peer_hook('claude', 'stop', {}, cfg=t), (0, '', ''))
    assert 'KeyError' in (Path(t)/'hooks.log').read_text()